In [ ]:
import os
import time

import bibtexparser
import flatdict as fd
import numpy as np
import pandas as pd
import requests

from pynxtools_apm.examples.oasisb_bibliography import get_bibliographical_metadata
from pynxtools_apm.examples.oasisb_openalex import get_data_for_doi_from_openalex
from pynxtools_apm.examples.oasisb_utils import get_project_id

rng = np.random.default_rng(seed=42)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype="str",
).fillna("")
with open(f"{src_directory}{os.sep}aaa_legacy_data.bib") as fp:
    bib = bibtexparser.load(fp).entries_dict
project_range: tuple[int, int] = (1, 155)

## Query for each project bibliographical metadata from OpenAlex

Querying metadata in addition to the DOI allows to cross-check authors, institutions, and copyright relevant details.

In [ ]:
api_queries_cnt = 0
api_queries_max = 10
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "":
        # row.parse == 2 if really only cross-ref data if available if dataset is CC0-1.0 or CC-BY-4.0
        # row.parse in (1, 2) if also allowing locally shared datasets, these will not be uploaded to any public deployment though
        if project_range[0] <= int(row.id) <= project_range[1]:
            project_id = get_project_id(f"{row.id}")
            data_and_paper = get_bibliographical_metadata(bib, row.project_name)

            n_queries = get_data_for_doi_from_openalex(bib, data_and_paper)

            api_queries_cnt += n_queries  # sleep only when necessary
            if api_queries_cnt >= api_queries_max:
                sleep = float(rng.uniform(1, 10))
                print(f"Sleeping for {sleep}s")
                time.sleep(sleep)
                api_queries_cnt = 0
print("Batch querying queue done")

***

## Assure that we have for each dataset copyright information and assure that it matches with those from OpenAlex

In [ ]:
# https://spdx.org/licenses
# https://developers.openalex.org
# build a look-up table of bib keys
import webbrowser
from json import JSONDecodeError

for row in spread_sheet_of_all_projects.itertuples(index=True):
    # if row.parse in (1, 2):  # get cross-ref data if available if dataset is CC0 or CC-BY-4.0
    if int(row.project_name) < 1:  # skip all before
        continue
    if int(row.project_name) > 822:  # skip all after
        continue
    if row.license != "":
        continue
    project_id = get_project_id(f"{row.project_name}")
    match = []
    token = f"D{project_id}"
    for key in bib:
        if not key.startswith(token):
            continue
        if key not in match:
            match = [key]
        else:
            print(f"ERROR::Multiple matching D{project_id}* keys")
    if len(match) != 1:
        pass
        # print(f"ERROR::D{project_id}* not found")
    else:
        # print(f"{project_id}, {match[0]}")
        if "doi" in bib[match[0]]:
            doi = bib[match[0]]["doi"]
            webbrowser.open_new_tab(f"https://doi.org/{doi}")
            continue
            if doi.startswith("10.5281"):
                url = f"https://zenodo.org/api/records/{doi.replace('10.5281/zenodo.', '')}"
                try:
                    data = requests.get(url, timeout=30).json()
                    print(
                        f"{project_id}, {match[0]}, {doi}, {data['metadata']['license']['id']}"
                    )
                    del data
                except (JSONDecodeError, KeyError):
                    pass
                # time.sleep(1.0)
                # Zenodo anonymous API access rate limit 60/min, currently code though is slower than 1s per project
                del url
        else:
            pass
            # print(f"{project_id}, no DOI")
    del project_id, match, token
    continue

    if row.license not in ("CC BY 4.0", "CC0 1.0", "CC BY 4.0 / CC0 1.0"):
        pass
        # search matching reference from bib, take advantage that each reference always either startswith
        # f"D{project_id}" (dataset) or f"A{project_id}" (associated key paper)

In [ ]:
json_files = set()
for root, dirs, files in os.walk("openalex"):
    for file in files:
        # fpath = f"{root}/{file}".replace(os.sep * 2, os.sep)
        json_files.add(file)
print(len(json_files))

In [ ]:
mdata = fd.FlatDict(data, delimiter="/")
for key, obj in mdata.items():
    print(f"{key}, {obj}")

In [ ]:
authorships = data.get("authorships", [])
if len(authorships) == 0:
    print("No authorships")

first_author = authorships[0]

institutions = first_author.get("institutions", [])
if not institutions:
    print("No institutions")

first_institution = institutions[0]

result = {
    "doi": doi,
    "first_author": first_author.get("author", {}).get("display_name"),
    "institution": first_institution.get("display_name"),
    "country_code_iso3": first_institution.get("country_code"),
    "town": first_institution.get("city"),
}
print(result)